# maths-tools pipeline

This notebook experiments with different strategies for maths specific questions. Mainly it builds the pipeline around the following 4 arms:
1. Direct (no CoT): Uses the P0_BASELINE strategy directly
2. CoT, no tools: Leverages CoT, no tools usage
3. CoT + det. tools: Leverages CoT, uses tools, but also tries to reduce tool usage via a cheap regex gate
4. CoT + LLM tools : Uses tools along with CoT, no gate


# Libraries

In [1]:
!pip -q install transformers accelerate bitsandbytes sentencepiece
!pip -q install mistral-common --upgrade
!pip install --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [2]:
# this might be needed depending on chosen model and environment
# uncomment if you see relevant error and restart kernel after executing this cell (crucial)
#!pip -q install "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main"

In [3]:
import time
import torch
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
    Mistral3ForConditionalGeneration
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import sys
import os
import re
from tqdm import tqdm
from dataclasses import dataclass
from typing import Callable, Union, Optional, List
from collections import Counter
import shutil
import requests
from urllib.parse import quote
from dataclasses import replace

import re
import time
import torch
import math
import sympy as sp
from sympy import symbols, sympify, Rational, simplify, nsimplify


In [4]:
import os
print(os.listdir('/kaggle/input/datasets'))

['isemzden']


In [5]:
# Fetch the secret from Kaggle's vault
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
# Log in to Hugging Face
login(token=hf_token)

# polimillionaire client

# Define the path to the directory containing your package
package_parent_dir = '/kaggle/input/datasets/isemzden/millionaire-client'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

from millionaire_client import MillionaireClient, AuthenticationError
pwd = user_secrets.get_secret("poli-millionaire")

API_URL = "http://131.175.15.22:51111/"
username = "cisemozden"
password = pwd

client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

['/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/kaggle/input/datasets/isemzden/millionaire-client']

Welcome, cisemozden! (Role: student)


# Model Configuration

In [6]:
# model family helpers

def is_gemma3(model_name):

    model_name = model_name.lower()

    return "gemma-3" in model_name


def is_gemma2(model_name):

    model_name = model_name.lower()

    return "gemma-2" in model_name

def is_gemma4(model_name):

    model_name = model_name.lower()

    return "gemma-4" in model_name


def is_mistral3(model_name):

    model_name = model_name.lower()

    return "mistral-small-3" in model_name or "magistral" in model_name

In [7]:
MODEL_NAME = "google/gemma-4-E4B-it"

# The below models work for our project without any fine tuning
# Best-performing models tested so far marked with ★
# Qwen:    "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-14B-Instruct" ★, "Qwen/Qwen3-4B-Instruct-2507"
# Mistral: "mistralai/Mistral-7B-Instruct-v0.3", "mistralai/Ministral-3-8B-Instruct-2512", "mistralai/Mistral-Small-3.2-24B-Instruct-2506" ★
# Gemma:   "google/gemma-2-9b-it", "google/gemma-3-4b-it", "google/gemma-3-12b-it", "google/gemma-4-E4B-it" ★
# Llama:   "meta-llama/Llama-3.1-8B-Instruct", "meta-llama/Llama-3.2-3B-Instruct"

GENERATION_CONFIG = {
    "max_new_tokens": 8,           # number is enough; reduces latency
    "do_sample": False,            # greedy; deterministic for fair comparisons
    "use_cache": True,             # set to True if model fits memory easily but is slow
    "output_scores": True,         # logits at every gen step (for confidence)
    "return_dict_in_generate": True,
}

USE_QUANTIZATION = True            # set to true if model very large

# Latency we consider "risky" given the 30s server timeout
LATENCY_DANGER_THRESHOLD_S = 20.0

# Where to store the offline question bank
QUESTION_BANK_PATH = "/kaggle/working/question_bank.json"
#QUESTION_BANK_PATH_EVAL = "/kaggle/input/millionaire-client/question_bank.json"
QUESTION_BANK_PATH_EVAL = "/kaggle/input/datasets/isemzden/millionaire-client/question_bank.json"

In [8]:
# quantization (4-bit) and model args

if USE_QUANTIZATION:

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model_kwargs = {
        "quantization_config": bnb_config,
        "device_map": "auto",
        "dtype": torch.float16,
    }

else:

    model_kwargs = {
        "device_map": "auto",
        "dtype": torch.float16,
    }

In [9]:
# load/swap model (and tokenizer where needed)

# paths for models mounted via Kaggle Models / Datasets
# set to None (or omit) for models we want to always fetch from HF hub
KAGGLE_MODEL_PATHS = {
    "Qwen/Qwen2.5-14B-Instruct":
        "/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct/1",
    "google/gemma-4-E4B-it":
        "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1",
}


def _resolve_model_path(model_name):
    """
    Return (path_or_name, source_label).
    Prefers a Kaggle-mounted local copy when present; otherwise returns the
    HF hub name so transformers will download (and cache) it.
    """
    local = KAGGLE_MODEL_PATHS.get(model_name)
    if local and Path(local).exists():
        return local, "kaggle"
    return model_name, "hub"


def load_model_globally(model_name):
    """
    Load (or swap to) a model.
    - Frees GPU memory and globals of any currently loaded model.
    - Uses /kaggle/input/... when available, else HF hub.
    - HF hub uses its on-disk cache automatically, so a previously-downloaded
      model is reloaded for free on a later swap-back (no re-download).
    """
    global MODEL_NAME, model, tokenizer, processor, mistral_tokenizer
    import gc

    # Free GPU memory + globals of whatever was loaded before
    for var in ("model", "tokenizer", "processor", "mistral_tokenizer"):
        if var in globals() and globals()[var] is not None:
            globals()[var] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    MODEL_NAME = model_name
    source_path, source = _resolve_model_path(model_name)
    print(f"Loading {model_name}  (source: {source}, path: {source_path})")

    if is_mistral3(MODEL_NAME):
        # Mistral's chat tokenizer is separate. If we have a local copy,
        # try loading the tokenizer file from it; else fall back to hub.
        mistral_tokenizer = None
        if source == "kaggle":
            for tk_name in ("tekken.json", "tokenizer.json"):
                tk_path = Path(source_path) / tk_name
                if tk_path.exists():
                    mistral_tokenizer = MistralTokenizer.from_file(str(tk_path))
                    break
        if mistral_tokenizer is None:
            mistral_tokenizer = MistralTokenizer.from_hf_hub(MODEL_NAME)

        model = Mistral3ForConditionalGeneration.from_pretrained(
            source_path, **model_kwargs
        ).eval()
        tokenizer = None
        processor = None

    elif is_gemma3(MODEL_NAME):
        processor = AutoProcessor.from_pretrained(source_path)
        model = Gemma3ForConditionalGeneration.from_pretrained(
            source_path, **model_kwargs
        ).eval()
        tokenizer = None
        mistral_tokenizer = None
    elif is_gemma4(MODEL_NAME):
        # 1. Load the processor to fix the 'NoneType' error
        processor = AutoProcessor.from_pretrained(source_path)
        
        # 2. FIX: Use AutoModelForCausalLM so it dynamically builds the correct architecture 
        # instead of forcing the Gemma 3 multimodal wrapper!
        model = AutoModelForCausalLM.from_pretrained(
            source_path, trust_remote_code=True, **model_kwargs
        ).eval()
        tokenizer = None
        mistral_tokenizer = None
    else:
        tokenizer = AutoTokenizer.from_pretrained(source_path, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            source_path, trust_remote_code=True, **model_kwargs
        )
        processor = None
        mistral_tokenizer = None

    print(f"  loaded: {model_name}")
    return model

In [10]:
model = load_model_globally(MODEL_NAME)

Loading google/gemma-4-E4B-it  (source: kaggle, path: /kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1)


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

  loaded: google/gemma-4-E4B-it


# Prompt Engineering

In [ ]:
from dataclasses import dataclass, field
from typing import Union, Callable, Optional, Any

@dataclass
class PromptStrategy:
    """
    Bundles everything needed to evaluate a prompting approach.
    """
    name: str
    system_prompt: Union[str, Callable[[Optional[int]], str]]
    user_builder: Callable[[str, List[str], Optional[int]], str]
    parser: Callable[[str, dict], Optional[int]]
    max_new_tokens: int = 8
    n_samples: int = 1
    do_sample: bool = False
    temperature: float = 0.0
    use_rag: bool = False
    tools: List[Any] = field(default_factory=list)
    

    def get_system(self, competition_id=None):
        if callable(self.system_prompt):
            return self.system_prompt(competition_id)
        return self.system_prompt


# Parsers

def parse_direct_digit(raw_output, confidence):
    """
    For direct-answer prompts. Regex for first 1-4, fallback to logit argmax.
    """
    m = re.search(r"[1-4]", raw_output.strip())
    if m:
        return int(m.group())
    if confidence:
        return max(confidence, key=confidence.get)
    return None


def parse_answer_line(raw_output, confidence):
    """
    For CoT-style outputs. Prefers explicit 'Answer: N' markers.
    """
    text = raw_output.strip()
    explicit_patterns = [
        r"(?:final\s+answer|the\s+answer\s+is|answer)\s*[:\-]?\s*([1-4])",
        r"(?:i\s+(?:choose|pick|select))\s+(?:option\s+)?([1-4])",
        r"option\s+([1-4])",
    ]
    for pat in explicit_patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return int(m.group(1))
    # Bare digit at end of text
    m = re.search(r"([1-4])\s*[\.\)]?\s*$", text)
    if m:
        return int(m.group(1))
    # Last digit anywhere
    digits = re.findall(r"[1-4]", text)
    if digits:
        return int(digits[-1])
    if confidence:
        return max(confidence, key=confidence.get)
    return None


def parse_eliminate(raw_output, confidence):
    """
    For P5 elimination format: 'N: yes/no'. Falls through to parse_answer_line.
    """
    text = raw_output.strip()
    # Explicit Answer: line wins
    m = re.search(r"answer\s*[:\-]?\s*([1-4])", text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    # Affirmative marker on a numbered option
    positive = r"(?:yes|correct|possible|right|true|likely|plausible)"
    m = re.search(rf"([1-4])\s*[:.\-]\s*{positive}\b", text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return parse_answer_line(text, confidence)


# User-prompt builders

def _user_builder_basic(question, options, competition_id=None):
    return (
        f"Question:\n{question}\n\n"
        f"Options:\n"
        f"1. {options[0]}\n"
        f"2. {options[1]}\n"
        f"3. {options[2]}\n"
        f"4. {options[3]}\n\n"
        f"Answer:"
    )


# P0: direct-answer baseline

P0_BASELINE = PromptStrategy(
    name="P0_baseline",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "You will receive one question and four numbered options.\n"
        "Choose the single most likely correct option.\n\n"
        "Output format:\n"
        "- Reply with ONLY one digit: 1, 2, 3, or 4.\n"
        "- No words, no punctuation, no explanation.\n"
        "- If unsure, still pick your best guess. You cannot skip."
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=512,
)

P0_BASELINE_try = PromptStrategy(
    name="P0_baseline",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "You will receive one question and four numbered options.\n"
        "Choose the single most likely correct option.\n\n"
        "Output format:\n"
        "- First briefly explain your reasoning along with the formulas.\n"
        "- Then reply with ONLY one digit: 1, 2, 3, or 4.\n"
        "- If unsure, still pick your best guess. You cannot skip."
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=512,
)


# P1: few-shot

P1_FEWSHOT = PromptStrategy(
    name="P1_fewshot",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n"
        "Reply with ONLY one digit (1, 2, 3, or 4). No explanation.\n\n"
        "Examples:\n\n"
        "Question: What is the chemical symbol for gold?\n"
        "Options:\n1. Au\n2. Ag\n3. Go\n4. Gd\nAnswer: 1\n\n"
        "Question: What is 12 multiplied by 7?\n"
        "Options:\n1. 72\n2. 84\n3. 96\n4. 108\nAnswer: 2\n\n"
        "Question: What is the largest planet in our solar system?\n"
        "Options:\n1. Earth\n2. Mars\n3. Jupiter\n4. Saturn\nAnswer: 3\n\n"
        "Question: In which year did World War II end?\n"
        "Options:\n1. 1943\n2. 1944\n3. 1946\n4. 1945\nAnswer: 4\n"
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=8,
)


# P2: chain-of-thought

P2_COT = PromptStrategy(
    name="P2_cot",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "Think briefly about which option is correct — at most 3 short sentences.\n"
        "Then on the final line write exactly:\n"
        "Answer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
)


# P3: CoT + self-consistency

P3_SELF_CONSISTENCY = PromptStrategy(
    name="P3_self_consistency",
    system_prompt=P2_COT.system_prompt,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
    n_samples=3,           # increase to 5 if accuracy boost justifies the latency
    do_sample=True,
    temperature=0.7,
)


# P4: per-competition specialization

_P4_SYSTEMS = {
    0: (  # Entertainment
        "You are an expert on entertainment trivia (movies, music, TV, "
        "games, books, celebrities). Recall the precise fact and reply "
        "with ONLY one digit (1, 2, 3, or 4)."
    ),
    1: (  # Ancient History and Politics
        "You are an expert on ancient history and politics. Consider "
        "dates, historical context, and political systems carefully. "
        "Reply with ONLY one digit (1, 2, 3, or 4)."
    ),
    2: (  # Science and Nature
        "You are an expert on science and nature. Apply established "
        "scientific principles and verified knowledge. Reply with ONLY "
        "one digit (1, 2, 3, or 4)."
    ),
    3: (  # Maths
        "You are an expert at multiple-choice mathematics problems.\n"
        "Work the problem step by step in 2-4 short lines.\n"
        "Then on the final line write exactly:\nAnswer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
}

def _p4_system(competition_id):
    return _P4_SYSTEMS.get(competition_id, P0_BASELINE.system_prompt)

P4_SPECIALIZED = PromptStrategy(
    name="P4_specialized",
    system_prompt=_p4_system,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
)


# P5: eliminate impossible options

P5_ELIMINATE = PromptStrategy(
    name="P5_eliminate",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "For each option, very briefly note whether it could be correct.\n"
        "Use the format 'N: yes/no — short reason' on one line per option.\n"
        "Then on the final line write exactly:\n"
        "Answer: N\n"
        "where N is 1, 2, 3, or 4.\n\n"
        "Example:\n"
        "1: no — wrong century\n"
        "2: yes — matches the description\n"
        "3: no — refers to a different event\n"
        "4: no — geographic mismatch\n"
        "Answer: 2"
    ),
    user_builder=_user_builder_basic,
    parser=parse_eliminate,
    max_new_tokens=200,
)


# Kept for reference only. Tried before with tool implemetation. Current version is with planner, P_PLANNER. 
P6_USE_TOOL = PromptStrategy(
    name="P6_use_tool",
    system_prompt=(
        "You are a multiple-choice maths expert.\n\n"
        "You will receive a question, four numbered options, and possibly a "
        "block of calculated results from tools. If tool results are provided, "
        "TRUST them and do NOT recompute by hand.\n\n"
        "Match the tool result to the correct option, then on the final line "
        "write exactly:\nAnswer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=512,
)

ALL_STRATEGIES = [P0_BASELINE, P1_FEWSHOT, P2_COT, P3_SELF_CONSISTENCY, P4_SPECIALIZED, P5_ELIMINATE, P6_USE_TOOL]
# RAG-augmented strategy variants — identical to the originals but with use_rag=True.
P0_RAG = replace(P0_BASELINE, name="P0_baseline_rag", use_rag=True)
P2_RAG = replace(P2_COT,      name="P2_cot_rag",      use_rag=True)
P4_RAG = replace(P4_SPECIALIZED, name="P4_specialized_rag", use_rag=True)
RAG_STRATEGIES = [P0_RAG, P2_RAG, P4_RAG]

# (A) A maths-only CoT system prompt -----------------------------------
# This is the prompt that was actually doing the work:
# letting the model reason, then forcing a single "Answer: N" line. We cap
# the tokens tightly (latency control) instead of the original 512.
_MATHS_COT_SYSTEM = (
    "You are an expert at multiple-choice mathematics.\n"
    "Reason in AT MOST 3 very short lines. Be terse.\n"
    "If a tool result block is provided, TRUST it and do not recompute.\n"
    "CRITICAL: your reasoning budget is limited, so you MUST commit to an "
    "answer quickly. As soon as you can, and ALWAYS before you run out of "
    "room, write the final line exactly as:\n"
    "Answer: N\n"
    "where N is 1, 2, 3, or 4. The 'Answer:' line is mandatory; never end "
    "without it. Output nothing after it."
)

P_MATHS_COT = PromptStrategy(
    name="P_maths_cot",
    system_prompt=_MATHS_COT_SYSTEM,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=160,        # tightened from 256: forces an early commit and
                               # keeps the hard cases under the 30s wall
)

# (B) A dedicated PLANNING system prompt
_PLANNER_SYSTEM = (
    "You decide whether a calculator tool is needed to answer a maths "
    "question. If a numeric computation (arithmetic, combinatorics, number "
    "theory, growth) is required, emit the appropriate tool call. If the "
    "question is conceptual/theoretical and needs no computation, emit no "
    "tool call at all. Do NOT answer the question yourself in this step."
)

P_PLANNER = PromptStrategy(
    name="P_planner",
    system_prompt=_PLANNER_SYSTEM,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=96,
)


# Tools

In [ ]:
def calculate_combinatorics(n: int, r: int, order_matters: bool):
    """
    Calculates the number of ways to choose 'r' items from 'n' items.
    
    Args:
        n: The total number of items to choose from.
        r: The number of items being chosen.
        order_matters: True for permutations (order matters), False for combinations (order does not matter).
    """
    if n < 0 or r < 0 or r > n:
        return {"error": "Invalid inputs. n and r must be positive, and r cannot be greater than n."}
    
    if order_matters:
        # Permutations: nPr = n! / (n-r)!
        result = math.perm(n, r)
        return {"operation": f"{n}P{r}", "result": result}
    else:
        # Combinations: nCr = n! / (r! * (n-r)!)
        result = math.comb(n, r)
        return {"operation": f"{n}C{r}", "result": result}

def calculate_normal_distribution(value: float = None, mean: float = None, std_dev: float = None, z_score: float = None):
    """
    Calculates statistics for a normal distribution. Provide either (value, mean, std_dev) 
    to get a z-score, or (z_score) to get the cumulative probability.
    
    Args:
        value: The observed value (X).
        mean: The population mean (μ).
        std_dev: The population standard deviation (σ).
        z_score: The calculated z-score.
    """
    if value is not None and mean is not None and std_dev is not None:
        if std_dev <= 0:
            return {"error": "Standard deviation must be greater than zero."}
        calculated_z = (value - mean) / std_dev
        # Calculate cumulative probability using the error function
        percentile = 0.5 * (1 + math.erf(calculated_z / math.sqrt(2)))
        return {
            "z_score": round(calculated_z, 4), 
            "percentile": round(percentile * 100, 2),
            "interpretation": f"The value {value} is {round(calculated_z, 2)} standard deviations from the mean."
        }
    
    elif z_score is not None:
        percentile = 0.5 * (1 + math.erf(z_score / math.sqrt(2)))
        return {"z_score": z_score, "percentile": round(percentile * 100, 2)}
        
    return {"error": "Insufficient arguments provided."}


def compute_math(expression: str):
    """
    Evaluates or simplifies a mathematical expression exactly using a symbolic
    math engine. Handles arithmetic with large numbers, fractions, powers,
    roots, logarithms, factorials, binomial coefficients, derivatives,
    integrals, summations, and limits. Returns both an exact and a decimal form.
 
    Use this for ANY calculation the question requires — do not compute by hand.
 
    Args:
        expression: A math expression in Python/SymPy syntax. Examples:
            "binomial(23, 6)"            -> combinations C(23,6)
            "factorial(5)"               -> 5!
            "(1 + sqrt(5))/2"            -> exact + decimal
            "log(2, 8)"                  -> log base 8 of 2
            "Rational(17, 40) * 160"     -> exact fraction arithmetic
            "diff(x + exp(x), x)"        -> derivative
            "integrate(x/(1 + x**3), (x, 0, oo))"  -> definite integral
            "summation(k, (k, 1, 100))"  -> sum
            "solve(x**2 + 2*x + 7 - (6*x + b), x)" (declare symbols in text)
    """
    try:
        # common single-letter symbols the model may use without declaring
        local = {s: symbols(s) for s in
                 ("x", "y", "z", "n", "k", "a", "b", "c", "r", "s", "t")}
        local.update({
            "binomial": sp.binomial, "factorial": sp.factorial,
            "sqrt": sp.sqrt, "log": sp.log, "ln": sp.log, "exp": sp.exp,
            "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "pi": sp.pi,
            "oo": sp.oo, "E": sp.E, "Rational": sp.Rational,
            "diff": sp.diff, "integrate": sp.integrate,
            "summation": sp.summation, "Sum": sp.Sum, "limit": sp.limit,
            "solve": sp.solve, "simplify": sp.simplify, "factor": sp.factor,
            "expand": sp.expand, "gcd": sp.gcd, "lcm": sp.lcm,
            "fibonacci": sp.fibonacci, "isprime": sp.isprime,
            "divisors": sp.divisors, "totient": sp.totient,
            "Abs": sp.Abs, "floor": sp.floor, "ceiling": sp.ceiling,
            "Mod": sp.Mod, "primerange": sp.primerange,
        })
        expr = eval(expression, {"__builtins__": {}}, local)  # restricted namespace
        exact = sp.simplify(expr) if not isinstance(expr, (list, tuple, dict)) else expr
        result = {"expression": expression, "exact": str(exact)}
        try:
            result["decimal"] = float(sp.N(exact))
        except Exception:
            result["decimal"] = None
        return result
    except Exception as e:
        return {"error": f"Could not evaluate '{expression}': {e}"}
 
 
def number_theory(operation: str, a: int, b: int = 0):
    """
    Integer / number-theory operations.
 
    Args:
        operation: One of "gcd", "lcm", "is_prime", "factorize",
            "divisors" (uses a only), "modulo", "factorial" (uses a only).
        a: Primary integer.
        b: Secondary integer (for gcd, lcm, modulo).
    """
    if operation == "gcd":
        return {"result": math.gcd(a, b)}
    if operation == "lcm":
        return {"result": math.lcm(a, b)}
    if operation == "is_prime":
        return {"is_prime": bool(sp.isprime(a))}
    if operation == "factorize":
        return {"factorization": {int(p): int(e) for p, e in sp.factorint(a).items()}}
    if operation == "divisors":
        d = [int(x) for x in sp.divisors(a)]
        return {"divisors": d, "count": len(d)}
    if operation == "modulo":
        if b == 0:
            return {"error": "Cannot mod by zero."}
        return {"result": a % b}
    if operation == "factorial":
        if a < 0:
            return {"error": "Undefined for negatives."}
        return {"result": math.factorial(a)}
    return {"error": f"Unknown operation '{operation}'."}
 
 
def compound_growth(principal: float = None, rate_percent: float = None,
                     years: float = None, final_amount: float = None):
    """
    Solves compound-growth problems final = principal * (1 + rate/100)**years.
    Provide exactly three of the four arguments; the tool solves for the fourth.
 
    Args:
        principal: Starting amount.
        rate_percent: Growth rate per period, in percent.
        years: Number of periods.
        final_amount: Ending amount.
    """
    given = [principal, rate_percent, years, final_amount]
    if sum(v is not None for v in given) != 3:
        return {"error": "Provide exactly three of four arguments."}
    try:
        if final_amount is None:
            return {"final_amount": round(principal * (1 + rate_percent/100)**years, 6)}
        if principal is None:
            return {"principal": round(final_amount / (1 + rate_percent/100)**years, 6)}
        if years is None:
            yrs = math.log(final_amount/principal) / math.log(1 + rate_percent/100)
            return {"years": round(yrs, 6)}
        if rate_percent is None:
            rate = ((final_amount/principal)**(1/years) - 1) * 100
            return {"rate_percent": round(rate, 6)}
    except Exception as e:
        return {"error": str(e)}
 
 
maths_tools = [
    compute_math,
    calculate_combinatorics,
    number_theory,
    compound_growth,
    calculate_normal_distribution
]

tools = maths_tools

# Pipeline & Wrapper

In [12]:
# whole message-to-llm builder

def build_messages(model_name, system_prompt, user_prompt):
    model_name = model_name.lower()

    # GEMMA 3 FORMAT
    if is_gemma3(model_name):
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt,}]},
            {"role": "user", "content": [{"type": "text", "text": user_prompt,}]}
        ]
        return messages

    # GEMMA 2 FORMAT
    elif is_gemma2(model_name):
        merged_prompt = (
            system_prompt.strip()
            + "\n\n"
            + user_prompt.strip()
        )
        messages = [
            {"role": "user", "content": merged_prompt,}
        ]
        return messages

    # STANDARD CHAT TEMPLATE MODELS
    else:
        messages = [
            {"role": "system", "content": system_prompt,},
            {"role": "user", "content": user_prompt,},
        ]
        return messages


# clean model output from text and logits

def extract_answer_from_text(text):
    """
    Parse the model's generated string for a 1-4 answer.
    Returns int in {1,2,3,4} or None if no digit found.
    """
    text = text.strip()
    match = re.search(r"[1-4]", text)
    return int(match.group()) if match else None


def extract_answer_from_logits(scores_step0, decode_fn, top_k=50):
    """
    Compute answer probabilities by inspecting the model's first-step logits.
    Walks the top-k most likely tokens at the first generation position and
    aggregates probability mass over those that decode to '1', '2', '3', '4'
    (stripped). This handles tokenizer quirks (leading-space tokens etc.)
    without needing per-tokenizer special cases.
    Returns:
        best_answer: int in {1,2,3,4} or None if no answer-token in top-k
        probs: dict {1: p1, 2: p2, 3: p3, 4: p4}, renormalized to sum to 1
    """
    probs = torch.softmax(scores_step0[0].float(), dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    answer_probs = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    for i in range(top_k):
        tok_id = top_ids[i].item()
        try:
            decoded = decode_fn(tok_id).strip()
        except Exception:
            continue
        if decoded in ("1", "2", "3", "4"):
            answer_probs[int(decoded)] += top_probs[i].item()

    total = sum(answer_probs.values())
    if total == 0:
        # No answer token found in the top-k; fall back to uniform
        return None, {k: 0.25 for k in (1, 2, 3, 4)}

    answer_probs = {k: v / total for k, v in answer_probs.items()}
    best = max(answer_probs, key=answer_probs.get)
    return best, answer_probs

In [ ]:
# SAFE ANSWER PARSER
def parse_answer_line_safe(raw_output, confidence):
    text = (raw_output or "").strip()

    explicit_patterns = [
        r"(?:final\s+answer|the\s+answer\s+is|answer)\s*[:\-]?\s*([1-4])",
        r"(?:i\s+(?:choose|pick|select))\s+(?:option\s+)?([1-4])",
        r"\boption\s+([1-4])\b",
    ]
    for pat in explicit_patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return int(m.group(1))
        
    if confidence:
        best = max(confidence, key=confidence.get)
        if abs(confidence[best] - 0.25) > 1e-6:
            return best

    m = re.search(r"(?:^|\n)\s*([1-4])\s*[\.\)]?\s*$", text)
    if m:
        return int(m.group(1))

    return None


def _decode_fn_for_model():
    """
    Return a unified decode_fn(tok_id) -> str for the active model family.
    """
    if is_mistral3(MODEL_NAME):
        return lambda tid: mistral_tokenizer.decode([tid])
    elif is_gemma3(MODEL_NAME):
        return lambda tid: processor.tokenizer.decode([tid])
    else:
        return lambda tid: tokenizer.decode([tid])


def extract_tool_calls(text):
    """Parse Gemma-4 native tool-call syntax: <|tool_call>call:name{k:v}<tool_call|>
    Unchanged from the official reference (matches the model's trained format)."""
    def cast(v):
        try: return int(v)
        except:
            try: return float(v)
            except: return {'true': True, 'false': False}.get(v.lower(), v.strip("'\""))
    pattern = r"(?:<\|tool_call>)?call:(\w+)\{(.*?)\}(?:<tool_call\|>)?"
    return [{
        "name": name,
        "arguments": {
            k: cast((v1 or v2).strip())
            for k, v1, v2 in re.findall(r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))', args)
        }
    } for name, args in re.findall(pattern, text, re.DOTALL)]


def _generate_with_messages(model, system_prompt, user_prompt, gen_config, tools,
                            keep_special_tokens=False):
    """
    One forward pass. Returns (raw_output, latency, first-step logits, messages).
    """
    messages = build_messages(MODEL_NAME, system_prompt, user_prompt)
    skip = not keep_special_tokens

    if is_mistral3(MODEL_NAME):
        tokenized = mistral_tokenizer.encode_chat_completion(
            ChatCompletionRequest(messages=messages)
        )
        input_ids = torch.tensor([tokenized.tokens], device=model.device)
        attention_mask = torch.ones_like(input_ids)
        gen_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pad_token_id": 2,
            **gen_config,
        }
        input_len = len(tokenized.tokens)
        decode_seq = lambda toks: mistral_tokenizer.decode(toks.tolist()).strip()

    elif is_gemma3(MODEL_NAME):
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to(model.device, dtype=torch.bfloat16)
        gen_kwargs = {**inputs, **gen_config}
        input_len = inputs["input_ids"].shape[-1]
        decode_seq = lambda toks: processor.decode(toks, skip_special_tokens=skip).strip()

    elif is_gemma4(MODEL_NAME):
        text = processor.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, return_tensors="pt").to(model.device)
        inputs.pop("mm_token_type_ids", None)
        gen_kwargs = {**inputs, **gen_config}
        input_len = inputs["input_ids"].shape[-1]
        decode_seq = lambda toks: processor.decode(toks, skip_special_tokens=skip).strip()
    else:
        prompt_str = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)
        gen_kwargs = {**inputs, "pad_token_id": tokenizer.eos_token_id, **gen_config}
        input_len = inputs["input_ids"].shape[1]
        decode_seq = lambda toks: tokenizer.decode(toks, skip_special_tokens=skip).strip()

    start = time.time()
    with torch.inference_mode():
        gen_out = model.generate(**gen_kwargs)
    latency = time.time() - start

    sequences = gen_out.sequences
    generated_tokens = sequences[0][input_len:]
    raw_output = decode_seq(generated_tokens)
    scores_step0 = gen_out.scores[0] if getattr(gen_out, "scores", None) else None
    return raw_output, latency, scores_step0, messages


def _format_tool_results_text(calls, results):
    """Turn executed tool calls + results into a plain-text block we can
    splice into the user prompt (model-agnostic, no template gymnastics)."""
    lines = ["Calculated results from tools:"]
    for c, r in zip(calls, results):
        args_str = ", ".join(f"{k}={v}" for k, v in c["arguments"].items())
        lines.append(f"- {c['name']}({args_str}) -> {r['response']}")
    return "\n".join(lines)


_COMBINATORICS_RE = re.compile(
    r"\b(how many ways|committees?|combinations?|permutations?|choose\b|"
    r"choosing|arrange(?:ment)?s?|distinct (?:committees|ways|arrangements))",
    re.I,
)
_NCHOOSE_RE = re.compile(r"choose\s+(\d+)\b.*?\bfrom\b.*?(\d+)", re.I | re.S)
_NORMAL_RE = re.compile(
    r"\b(standard deviation|z[-\s]?score|normally distributed|percentile|"
    r"mean of\b.*\bdeviation)", re.I,
)


def deterministic_tool_context(question, competition_id=None):
    """
    Try to answer the *computational* part of a question with a local tool,
    deterministically (no LLM). Returns (context_text, calls, results) or
    ("", [], []) if no rule fires. Math category only.
    """
    if competition_id not in (3, None):
        return "", [], []
    q = question

    if _COMBINATORICS_RE.search(q):
        m = _NCHOOSE_RE.search(q)
        if m:
            try:
                r_, n_ = int(m.group(1)), int(m.group(2))
                if 0 <= r_ <= n_:
                    res = calculate_combinatorics(n=n_, r=r_, order_matters=False)
                    calls = [{"name": "calculate_combinatorics",
                              "arguments": {"n": n_, "r": r_, "order_matters": False}}]
                    results = [{"name": "calculate_combinatorics", "response": res}]
                    return _format_tool_results_text(calls, results) + "\n\n---\n\n", calls, results
            except Exception:
                pass

    return "", [], []



def _gemma4_apply_and_generate(model, messages, gen_config, tools,
                               keep_special_tokens=False):
    """
    GEMMA-4 ONLY. Generate from a full `messages` list (which may include an
    assistant tool_calls turn and tool_responses), following the official
    reference: apply_chat_template(..., tools=tools) -> processor -> generate.
    Returns (raw_output, latency, scores_step0).
    """
    skip = not keep_special_tokens
    text = processor.apply_chat_template(
        messages, tools=tools, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    inputs.pop("mm_token_type_ids", None)
    gen_kwargs = {**inputs, **gen_config}
    input_len = inputs["input_ids"].shape[-1]

    start = time.time()
    with torch.inference_mode():
        gen_out = model.generate(**gen_kwargs)
    latency = time.time() - start

    generated_tokens = gen_out.sequences[0][input_len:]
    raw_output = processor.decode(generated_tokens, skip_special_tokens=skip).strip()
    scores_step0 = gen_out.scores[0] if getattr(gen_out, "scores", None) else None
    return raw_output, latency, scores_step0


def _execute_calls(calls):
    """Run parsed tool calls against the globally-defined tool functions.
    Returns the reference-shaped results list: [{name, response}, ...]."""
    results = []
    for c in calls:
        fn = globals().get(c["name"])
        if fn is None:
            results.append({"name": c["name"], "response": {"error": "unknown tool"}})
            continue
        try:
            results.append({"name": c["name"], "response": fn(**c["arguments"])})
        except Exception as e:
            results.append({"name": c["name"], "response": {"error": str(e)}})
    return results


def _gemma4_tool_turn(messages, calls, results):
    """Append the assistant tool-call turn exactly as the reference does, so the
    answer pass sees results in the trained `tool_responses` role rather than as
    free text spliced into the user message."""
    messages = list(messages)
    messages.append({
        "role": "assistant",
        "tool_calls": [{"function": c} for c in calls],
        "tool_responses": results,
    })
    return messages



def _force_answer_followup(model, system_prompt, user_prompt, prior_text,
                           decode_fn):
    """
    Recovery pass for when the main generation truncated before emitting
    'Answer: N' (the hard questions hit the token cap mid-reasoning and would
    otherwise parse to None). We replay the question + whatever reasoning was
    produced, then ask ONLY for the digit, capped at a handful of tokens.
    Returns (answer_int_or_None, confidence_dict, latency).
    """
    followup_user = (
        user_prompt
        + "\n\nReasoning so far:\n" + (prior_text or "").strip()
        + "\n\nBased on the above, give ONLY the final choice now.\n"
          "Answer: "
    )
    cfg = {
        "use_cache": True,
        "output_scores": True,
        "return_dict_in_generate": True,
        "max_new_tokens": 8,
        "do_sample": False,
    }
    raw, lat, scores0, _ = _generate_with_messages(
        model, system_prompt, followup_user, cfg, tools=None,
        keep_special_tokens=False,
    )
    _, conf = (extract_answer_from_logits(scores0, decode_fn)
               if scores0 is not None
               else (None, {k: 0.25 for k in (1, 2, 3, 4)}))
    ans = parse_answer_line_safe(raw, conf)
    if ans is None and conf:  # last resort: take argmax logit even if ~uniform
        ans = max(conf, key=conf.get)
    return ans, conf, lat


def ask_llm(model, question, option_texts, competition_id=None, strategy=None,
            use_rag=None, tools=[], use_tools=False, tool_mode="deterministic"):
    """
      - tool_mode="deterministic":
            a cheap regex gate may inject tool results; then ONE answer pass.
            No planning generation is ever spent.
      - tool_mode="llm":
            keeps the original LLM-decides planning pass (with the prompt/decode
            FIXES applied) followed by the answer pass. Use only when you
            explicitly want to study that route; it costs two decodes.
      - use_tools=False:
            a single answer pass, no tools. (Plain baseline / CoT.)
    """
    if strategy is None:
        strategy = P0_BASELINE
    if use_rag is None:
        use_rag = strategy.use_rag

    system_prompt = strategy.get_system(competition_id)
    user_prompt = strategy.user_builder(question, option_texts, competition_id)

    total_latency = 0.0
    rag_latency = 0.0
    context = ""
    tool_calls_made, tool_results_made = [], []

    gemma4 = is_gemma4(MODEL_NAME)
    gemma4_messages = None

    if use_tools and tools:
        if tool_mode == "deterministic":
            ctx, dcalls, dresults = deterministic_tool_context(question, competition_id)
            if dcalls:
                tool_calls_made, tool_results_made = dcalls, dresults
                if gemma4:
                    # feed results through the trained tool_responses turn
                    base_messages = build_messages(MODEL_NAME, system_prompt, user_prompt)
                    gemma4_messages = _gemma4_tool_turn(base_messages, dcalls, dresults)
                else:
                    context += ctx

        elif tool_mode == "llm":
            planner_strategy = globals().get("P_PLANNER", strategy)
            planner_system = planner_strategy.get_system(competition_id)
            planner_config = {
                "use_cache": True,
                "output_scores": False,
                "return_dict_in_generate": True,
                "max_new_tokens": planner_strategy.max_new_tokens,
                "do_sample": False,
            }

            if gemma4:
                # Gemma-4 native tool calling
                planner_messages = build_messages(MODEL_NAME, planner_system, user_prompt)
                raw_tool_output, step1_latency, _ = _gemma4_apply_and_generate(
                    model, planner_messages, planner_config, tools=tools,
                    keep_special_tokens=True,   # keep <|tool_call> delimiters
                )
                total_latency += step1_latency
                calls = extract_tool_calls(raw_tool_output)
                if calls:
                    results = _execute_calls(calls)
                    tool_calls_made, tool_results_made = calls, results
                    gemma4_messages = _gemma4_tool_turn(planner_messages, calls, results)
            else:
                # non-gemma4: original text-splice planner path
                raw_tool_output, step1_latency, _, _ = _generate_with_messages(
                    model, planner_system, user_prompt, planner_config, tools=tools,
                    keep_special_tokens=True,
                )
                total_latency += step1_latency
                calls = extract_tool_calls(raw_tool_output)
                if calls:
                    results = _execute_calls(calls)
                    tool_calls_made, tool_results_made = calls, results
                    context += _format_tool_results_text(calls, results) + "\n\n---\n\n"

    if use_rag:
        rag_start = time.time()
        retrieved_docs = retriever.retrieve_and_format(question)
        rag_latency = time.time() - rag_start
        if retrieved_docs:
            context += f"References:\n{retrieved_docs}\n\n---\n\n"

    if context:
        user_prompt = context + user_prompt

    gen_config = {
        "use_cache": True,
        "output_scores": True,
        "return_dict_in_generate": True,
        "max_new_tokens": strategy.max_new_tokens,
        "do_sample": strategy.do_sample,
    }
    if strategy.do_sample:
        gen_config["temperature"] = strategy.temperature

    decode_fn = _decode_fn_for_model()
    answers, raw_outputs, confidences = [], [], []

    for _ in range(strategy.n_samples):
        if gemma4_messages is not None:
            raw_output, latency, scores_step0 = _gemma4_apply_and_generate(
                model, gemma4_messages, gen_config, tools=tools,
                keep_special_tokens=False,
            )
        else:
            raw_output, latency, scores_step0, _ = _generate_with_messages(
                model, system_prompt, user_prompt, gen_config, tools=None,
                keep_special_tokens=False,   # answer text should be clean
            )
        total_latency += latency
        raw_outputs.append(raw_output)

        _, conf = (extract_answer_from_logits(scores_step0, decode_fn)
                   if scores_step0 is not None
                   else (None, {k: 0.25 for k in (1, 2, 3, 4)}))

        ans = parse_answer_line_safe(raw_output, conf)

        if ans is None and gemma4_messages is None:
            ans, conf, fb_lat = _force_answer_followup(
                model, system_prompt, user_prompt, raw_output, decode_fn
            )
            total_latency += fb_lat

        confidences.append(conf)
        answers.append(ans)

    if strategy.n_samples > 1:
        valid = [a for a in answers if a is not None]
        if valid:
            counts = Counter(valid)
            final_answer = counts.most_common(1)[0][0]
            total = sum(counts.values())
            final_confidence = {k: counts.get(k, 0) / total for k in (1, 2, 3, 4)}
        else:
            final_answer = None
            final_confidence = {k: 0.25 for k in (1, 2, 3, 4)}
        return {
            "answer": final_answer,
            "confidence": final_confidence,
            "raw_output": raw_outputs,
            "latency": total_latency,
            "rag_latency": rag_latency,
            "samples": answers,
            "tool_calls": tool_calls_made,
            "tool_results": tool_results_made,
        }

    return {
        "answer": answers[0],
        "confidence": confidences[0],
        "raw_output": raw_outputs[0],
        "latency": total_latency,
        "rag_latency": rag_latency,
        "samples": None,
        "tool_calls": tool_calls_made,
        "tool_results": tool_results_made,
    }


In [ ]:
# This smoke test now uses the strategies above.

# 1. This represents the raw data block from your question bank dictionary
question_entry = {
    "question": "A committee of 6 people must be chosen from a group of 23 candidates. How many distinct committees are possible?",
    "options": [
        "100947",
        "53130",
        "245157",
        "148764"
    ],
    "correct_text": "100947",   # C(23,6) = 100947
    "competition_id": 3,
    "level": 6,
}

# 2. Extract and assign to the target variables
test_question = question_entry["question"]
test_options = question_entry["options"]

category = question_entry["competition_id"]

# 3. Print them out to verify the format
print(f"test_question = {repr(test_question)}")
print(f"test_options = {test_options}")

for opt in range(len(test_options)):
    if test_options[opt] == question_entry["correct_text"]:
        correct_option_index = opt
        break


print(f"Testing Question: {test_question}")
print(f"Options: 1. {test_options[0]} | 2. {test_options[1]} | 3. {test_options[2]} | 4. {test_options[3]}\n")

# 2. Setup the configurations to compare
test_configs = [
    {"name": "A) DIRECT (no CoT)",        "strategy": P0_BASELINE, "use_tools": False, "tool_mode": "deterministic"},
    {"name": "B) CoT, no tools",          "strategy": P_MATHS_COT, "use_tools": False, "tool_mode": "deterministic"},
    {"name": "C) CoT + deterministic",    "strategy": P_MATHS_COT, "use_tools": True,  "tool_mode": "deterministic"},
    {"name": "D) CoT + LLM tools (gemma4)","strategy": P_MATHS_COT, "use_tools": True,  "tool_mode": "llm"},
]

# 3. Execute the single-turn runs
for config in test_configs:
    print("=" * 60)
    print(config["name"])
    print("=" * 60)
    
    # Run the question through your existing notebook wrapper
    result = ask_llm(
        model=model, 
        question=test_question, 
        option_texts=test_options,
        competition_id=3,          # 3 = Maths category
        strategy=config["strategy"],
        tools=maths_tools,
        use_rag=False,
        use_tools=config["use_tools"],
        tool_mode=config["tool_mode"],
    )
    
    # 4. Display the results cleanly
    predicted_digit = result["answer"]
    is_correct = (predicted_digit == correct_option_index + 1)
    status_marker = "✅ CORRECT" if is_correct else "❌ INCORRECT"
    
    print(f"Raw Output Text From Model:\n\"\"\"\n{result['raw_output'].strip()}\n\"\"\"\n")
    print(f"Parsed Answer Choice Digit : {predicted_digit} ({status_marker})")
    
    # Extract the highest probability assigned to the parsed token step
    if predicted_digit in result["confidence"]:
        conf_val = result["confidence"][predicted_digit] * 100
        print(f"Model Confidence           : {conf_val:.1f}%")
        
    fired = bool(result.get("tool_calls"))
    print(f"Tool fired                 : {fired}  {result.get('tool_calls')}")
    print(f"Execution Latency          : {result['latency']:.3f} seconds\n")

test_question = 'A committee of 6 people must be chosen from a group of 23 candidates. How many distinct committees are possible?'
test_options = ['100947', '53130', '245157', '148764']
Testing Question: A committee of 6 people must be chosen from a group of 23 candidates. How many distinct committees are possible?
Options: 1. 100947 | 2. 53130 | 3. 245157 | 4. 148764

A) DIRECT (no CoT)
Raw Output Text From Model:
"""
2
"""

Parsed Answer Choice Digit : 2 (❌ INCORRECT)
Model Confidence           : 25.0%
Tool fired                 : False  []
Execution Latency          : 1.074 seconds

B) CoT, no tools
Raw Output Text From Model:
"""
The problem asks for the number of ways to choose 6 people from 23, which is a combination $\binom{23}{6}$.
$\binom{23}{6} = \frac{23!}{6!(23-6)!} = \frac{23 \times 22 \times 21 \times 20 \times 19 \times 18}{6 \times 5 \times 4 \times 3 \times 2 \times 1} = 100947$.
Answer: 1
"""

Parsed Answer Choice Digit : 1 (✅ CORRECT)
Model Confidence           : 25.

# TEST

In [ ]:
# This version runs four arms:
#   A) direct  : single-digit answer, no reasoning      (P0_BASELINE)
#   B) cot     : capped CoT, no tools                    (P_MATHS_COT)
#   C) cot+det : same CoT + DETERMINISTIC tool fast-path (P_MATHS_COT, tools on)
#   D) cot+llm : same CoT + Gemma-4 NATIVE LLM tool-decide route (two decodes, expect higher latency)
import json, time, sys, io
from collections import Counter
from itertools import islice

MATHS_PATH = "/kaggle/input/datasets/isemzden/millionaire-client/maths.json"
LIMIT = 40

with open(MATHS_PATH, "r", encoding="utf-8") as f:
    bank = json.load(f)

def has_usable_answer(entry):
    ct = entry.get("correct_text")
    if ct in (None, "None", ""):
        return False
    return ct in entry.get("options", [])

usable_all = {qid: e for qid, e in bank.items() if has_usable_answer(e)}
usable = dict(islice(usable_all.items(), LIMIT))
skipped = len(bank) - len(usable_all)
print(f"Loaded {len(bank)} | usable: {len(usable_all)} | skipped: {skipped} | "
      f"evaluating: {len(usable)}\n")

def correct_digit(entry):
    ct = entry["correct_text"].strip()
    for i, opt in enumerate(entry["options"]):
        if opt.strip() == ct:
            return i + 1
    return None

# 3 arms
CONFIGS = [
    {"key": "direct",  "label": "A) direct (no CoT)",   "strategy": P0_BASELINE,
     "use_tools": False, "tool_mode": "deterministic"},
    {"key": "cot",     "label": "B) CoT, no tools",     "strategy": P_MATHS_COT,
     "use_tools": False, "tool_mode": "deterministic"},
    {"key": "cot_det", "label": "C) CoT + det. tools",  "strategy": P_MATHS_COT,
     "use_tools": True,  "tool_mode": "deterministic"},
    # D) only meaningful on Gemma-4 (native tool-calling).
    {"key": "cot_llm", "label": "D) CoT + LLM tools",   "strategy": P_MATHS_COT,
     "use_tools": True,  "tool_mode": "llm"},
]

records = []
totals = {c["key"]: {"correct": 0, "answered": 0, "latency": 0.0,
                     "max_latency": 0.0, "over_30s": 0} for c in CONFIGS}

for n, (qid, entry) in enumerate(usable.items(), 1):
    question = entry["question"]
    options = entry["options"]
    gold = correct_digit(entry)
    comp_id = entry.get("competition_id")

    print(f"\n--- Q {n}/{len(usable)} (qid={qid}, level={entry.get('level')}) ---", flush=True)
    print(f"    Q: {question[:80]}", flush=True)
    print(f"    Gold: option {gold}", flush=True)

    row = {"qid": qid, "level": entry.get("level"), "gold": gold,
           "question": question[:70]}

    for cfg in CONFIGS:
        try:
            res = ask_llm(
                model=model,
                question=question,
                option_texts=options,
                competition_id=comp_id,
                strategy=cfg["strategy"],
                tools=maths_tools,
                use_rag=False,
                use_tools=cfg["use_tools"],
                tool_mode=cfg["tool_mode"],
            )
            pred = res["answer"]
            lat = res["latency"]
            fired = bool(res.get("tool_calls"))
        except Exception as e:
            print(f"    [!] {cfg['key']} failed on {qid}: {e}", flush=True)
            pred, lat, fired = None, 0.0, False

        is_correct = (pred is not None and pred == gold)
        row[f"{cfg['key']}_pred"] = pred
        row[f"{cfg['key']}_correct"] = is_correct
        row[f"{cfg['key']}_latency"] = lat
        row[f"{cfg['key']}_tool_fired"] = fired

        t = totals[cfg["key"]]
        t["latency"] += lat
        t["max_latency"] = max(t["max_latency"], lat)
        if lat > 30.0:
            t["over_30s"] += 1
        if pred is not None:
            t["answered"] += 1
        if is_correct:
            t["correct"] += 1

        verdict = "CORRECT" if is_correct else "INCORRECT"
        tflag = " [tool]" if fired else ""
        print(f"    {cfg['label']:<22} pred=opt {pred} -> {verdict} ({lat:.2f}s){tflag}",
              flush=True)

    records.append(row)

N = len(usable)
print("\n" + "=" * 78)
print(f"{'Config':<24}{'Correct':>10}{'Acc':>8}{'Avg lat':>11}{'Max lat':>11}{'>30s':>7}")
print("-" * 78)
for cfg in CONFIGS:
    t = totals[cfg["key"]]
    acc = t["correct"] / N * 100 if N else 0
    avg_lat = t["latency"] / N if N else 0
    print(f"{cfg['label']:<24}{t['correct']:>4}/{N:<5}{acc:>6.1f}%"
          f"{avg_lat:>10.2f}s{t['max_latency']:>10.2f}s{t['over_30s']:>7}")
print("=" * 78)

tool_fixed = [r for r in records if r["cot_det_correct"] and not r["cot_correct"]]
tool_broke = [r for r in records if r["cot_correct"] and not r["cot_det_correct"]]
cot_gain   = [r for r in records if r["cot_correct"] and not r["direct_correct"]]
print(f"\nCoT effect  (B fixed over A): +{len(cot_gain)}")
print(f"Tool effect (C fixed over B): {len(tool_fixed) - len(tool_broke):+d} "
      f"(fixed {len(tool_fixed)}, broke {len(tool_broke)})")
n_fired = sum(r['cot_det_tool_fired'] for r in records)
print(f"Deterministic tool fired on: {n_fired}/{N} questions")

out_path = "/kaggle/working/maths_3arm_comparison.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({"totals": totals, "records": records}, f, indent=2, ensure_ascii=False)
print(f"\nPer-question detail saved to {out_path}")


Loaded 210 | usable: 112 | skipped: 98 | evaluating: 40


--- Q 1/40 (qid=67ace252271f13bd, level=1) ---
    Q: In a group of 10 scores, the largest score is increased by 40 points. What will 
    Gold: option 4
    A) direct (no CoT)     pred=opt 2 -> INCORRECT (0.37s)
    B) CoT, no tools       pred=opt 4 -> CORRECT (12.68s)
    C) CoT + det. tools    pred=opt 4 -> CORRECT (12.75s)
    D) CoT + LLM tools     pred=opt 4 -> CORRECT (16.39s)

--- Q 2/40 (qid=fe641cb7bb8d112b, level=2) ---
    Q: The map x -> axa^2 of a group G into itself is a homomorphism if and only if
    Gold: option 3
    A) direct (no CoT)     pred=opt 2 -> INCORRECT (0.37s)
    B) CoT, no tools       pred=opt 1 -> INCORRECT (26.70s)
    C) CoT + det. tools    pred=opt 1 -> INCORRECT (26.27s)
    D) CoT + LLM tools     pred=opt 1 -> INCORRECT (33.78s)

--- Q 3/40 (qid=da08d9eece2bf712, level=1) ---
    Q: The region bounded by the x-axis and the function f(x) = x/(1 + x^3) is rotated 
    Gold: option 2
    A) dir